In [224]:
import pandas as pd 
import numpy as np
from keras.models import Sequential
from keras.layers import Dense
from tensorflow import keras
from tensorflow.keras import layers

In [190]:
df = pd.read_csv('kalshi_initial.csv')
df['open_time'] = pd.to_datetime(df['open_time'])
df['curr_time'] = pd.to_datetime(df['curr_time'])
df['close_time'] = pd.to_datetime(df['close_time'])
df['time_to_close'] = (df['close_time'] - df['curr_time']).dt.total_seconds()
df = df.sort_values(by = ['coin', 'open_time', 'curr_time'])
df = df.drop(columns=['id', 'strike_type'])
# df = df.set_index('curr_time')
df = df.set_index('curr_time')


In [230]:
def build_coin_df(coin): 
    global df 
    df_copy = df[df['coin'] == coin].copy()
    col_identifiers = ['open_time', 'close_time']
    df_copy['next_price_dollars_lead1'] = df_copy.groupby(col_identifiers)['last_price_dollars'].shift(-5)
    df_copy = df_copy[df_copy['time_to_close'] > -0.01]

    df_copy = df_copy.add_prefix(f'{coin}_')
    
    return df_copy

In [231]:
lst = []
for i in ['BTC', 'ETH', 'XRP', 'SOL']: 
    lst.append(build_coin_df(i))

combined = pd.concat(lst, join='inner', axis=1)
# combined = combined.dropna(axis='columns')
combined.dropna(subset=['BTC_next_price_dollars_lead1'] ,inplace=True)
combined = combined.drop(columns=['BTC_coin', 'ETH_coin', 'XRP_coin', 'SOL_coin'])
time_to_close_col = combined['BTC_time_to_close']
combined = combined.select_dtypes(exclude='datetime64[us, UTC]')
combined.info() 

<class 'pandas.DataFrame'>
DatetimeIndex: 2364 entries, 2026-03-13 05:42:39.686553+00:00 to 2026-03-13 08:36:32.854934+00:00
Data columns (total 52 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   BTC_last_price_dollars        2364 non-null   float64
 1   BTC_liquidity_dollars         2364 non-null   float64
 2   BTC_no_ask_dollars            2364 non-null   float64
 3   BTC_no_bid_dollars            2364 non-null   float64
 4   BTC_yes_ask_dollars           2364 non-null   float64
 5   BTC_yes_bid_dollars           2364 non-null   float64
 6   BTC_open_interest             0 non-null      float64
 7   BTC_floor_strike              2364 non-null   float64
 8   BTC_cap_strike                0 non-null      float64
 9   BTC_volume                    0 non-null      float64
 10  BTC_volume_24h_fp             2364 non-null   float64
 11  BTC_time_to_close             2364 non-null   float64
 12  BTC_next_pr

In [232]:
y = combined['BTC_next_price_dollars_lead1'].values
X = combined.drop(columns=['BTC_next_price_dollars_lead1']).dropna(axis='columns')
X = pd.concat([X,time_to_close_col], axis=1).values

In [233]:
n = int(X.shape[0]*0.8)

X_train, X_test = X[:n], X[n:]
y_train, y_test = y[:n], y[n:]

In [234]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_train)

In [250]:
model = keras.Sequential([
    layers.Dense(64, activation="relu", input_shape=(X_train.shape[1],)),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation='sigmoid')  # regression output
])

/Users/erxcshi/courses/ml1grad/crypto_predictions/.venv/lib/python3.13/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [257]:
model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

In [258]:
history = model.fit(
    X_train,
    y_train,
    epochs=50,
)

Epoch 1/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 1s 571us/step - loss: 0.0035 - mae: 0.0387 
Epoch 2/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 549us/step - loss: 0.0027 - mae: 0.0323
Epoch 3/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 482us/step - loss: 0.0023 - mae: 0.0302
Epoch 4/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 494us/step - loss: 0.0020 - mae: 0.0284
Epoch 5/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 608us/step - loss: 0.0019 - mae: 0.0273
Epoch 6/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 513us/step - loss: 0.0019 - mae: 0.0270
Epoch 7/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 482us/step - loss: 0.0019 - mae: 0.0277
Epoch 8/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 485us/step - loss: 0.0017 - mae: 0.0259
Epoch 9/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 488us/step - loss: 0.0017 - mae: 0.0262
Epoch 10/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 496us/step - loss: 0.0016 - mae: 0.0244
Epoch 11/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 491us/step - loss: 0.0016 - mae: 0.0247
Epoch 12/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 569us/step - loss: 0.0016 - mae: 0.0248
Epoch 13/50
60/60 ━━━━━━

In [ ]:
pred = model.predict(X_test)
pred

60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 409us/step


array([[1.],
       [1.],
       [1.],
       ...,
       [1.],
       [1.],
       [1.]], shape=(1891, 1), dtype=float32)

: 

: 

In [ ]:
np.unique(pred)
